# protein-selector DB explorer

A static notebook for browsing `cache/protein_selector.db` -- no live dashboard,
just pandas over the SQLite cache. All real logic lives in the package
(`core/db.py`, each domain's `store.py`, `core/report.py`) -- this notebook only
calls into it, so there's nothing here worth unit-testing separately.

Two ways to look at the data, both shown below:
1. **Raw tables** -- any table, straight from SQLite via `pandas.read_sql_query`.
   Fastest way to eyeball what one stage actually persisted.
2. **The joined report** -- `core.report.build_report_table`, the same join
   `write_report_csv` uses. One row per candidate, every stage combined,
   per-exercise difficulty/tier/status included.

Requires the `notebook` dependency group: `uv sync --group notebook`.

In [ ]:
from pathlib import Path

import pandas as pd

from protein_selector.core.db import DEFAULT_DB_PATH
from protein_selector.core.report import build_report_table, rows_to_dataframe

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Point this at whichever db you want to inspect. DEFAULT_DB_PATH
# ("cache/protein_selector.db") is relative, so it depends on where the
# kernel's cwd is -- Jupyter usually launches with cwd = the notebook's own
# directory (notebooks/), not the repo root. Try repo-root-relative first,
# then one directory up (covers launching from notebooks/), then just fall
# back to DEFAULT_DB_PATH so the assert below gives a clear error instead of
# silently picking the wrong file.
_candidates = [DEFAULT_DB_PATH, Path("..") / DEFAULT_DB_PATH]
DB_PATH = next((p for p in _candidates if p.exists()), DEFAULT_DB_PATH)
assert DB_PATH.exists(), f"no db at {DB_PATH} -- run the pipeline first, or set DB_PATH above"
DB_PATH

## 1. What tables exist, and how big are they?

Generic -- works regardless of which stages have actually been run against this db.

In [ ]:
import sqlite3


def list_tables(db_path: Path) -> pd.DataFrame:
    """List every table in the db and its row count -- a quick 'what's actually in here' check."""
    with sqlite3.connect(db_path) as conn:
        table_names = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
        )["name"]
        return pd.DataFrame(
            {
                "table": table_names,
                "n_rows": [
                    conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
                    for name in table_names
                ],
            }
        )


list_tables(DB_PATH)

## 2. Any raw table, straight from SQLite

Change `TABLE_NAME` below to any name from the list above (`candidates`,
`simulability`, `parameterizability`, `pocket_detection`, `literature`,
`alphafold_entries`, `validation`, ...) -- no need to know each domain's
specific `load_*` function or dataclass just to look at it.

In [ ]:
def load_table(db_path: Path, table_name: str) -> pd.DataFrame:
    """Load one table as a DataFrame -- JSON-encoded columns (reasons/notes/pockets/etc.) stay as raw strings."""
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(f"SELECT * FROM {table_name}", conn)


TABLE_NAME = "candidates"
load_table(DB_PATH, TABLE_NAME)

In [ ]:
load_table(DB_PATH, "validation")

## 3. The joined report (one row per candidate, every stage combined)

This is the same function `write_report_csv` uses -- read-only, never calls a
network API, purely joins whatever is already persisted in `DB_PATH`.
Per-exercise columns are flattened with an `ex0X_` prefix
(`ex02_status`, `ex03_predicted_difficulty`, `ex04_tier`, ...).

In [ ]:
rows = build_report_table(db_path=DB_PATH)
report_df = rows_to_dataframe(rows)
report_df

## 4. A few common questions, as one-line filters/sorts

Adjust freely -- `report_df` is a plain DataFrame, nothing special about it.

In [ ]:
# Which candidates are actually suitable for at least one exercise?
report_df[report_df["suitable_for"] != "[]"][["pdb_id", "suitable_for"]]

In [ ]:
# The highest-value catch PLAN.md §5 calls out: looked easy (low predicted) but
# actually failed/was hard (positive gap) -- sorted worst-surprise first, per exercise.
for exercise in ("ex02", "ex03", "ex04"):
    gap_col = f"{exercise}_gap"
    surprises = report_df[report_df[gap_col].notna()].sort_values(gap_col, ascending=False)
    if not surprises.empty:
        print(f"--- {exercise}: predicted-vs-measured gap (looked-easier-than-it-was first) ---")
        display(surprises[["pdb_id", f"{exercise}_predicted_difficulty", f"{exercise}_measured_difficulty", gap_col, f"{exercise}_status"]])

In [ ]:
# Failures, with their failure_mode and the real validator notes (rationale_json holds the rest).
import json

for exercise in ("ex02", "ex03", "ex04"):
    failures = report_df[report_df[f"{exercise}_status"] == "fail"]
    if not failures.empty:
        print(f"--- {exercise} failures ---")
        for _, row in failures.iterrows():
            notes = json.loads(row["rationale_json"])[exercise]
            print(f"{row['pdb_id']}: {row[f'{exercise}_failure_mode']} -- {notes}")

In [ ]:
# Spiral-curriculum tiers per exercise (PLAN.md §5) -- how many candidates land in each bucket.
for exercise in ("ex02", "ex03", "ex04"):
    counts = report_df[f"{exercise}_tier"].value_counts(dropna=False)
    print(f"{exercise}: {counts.to_dict()}")

## 5. Writing a fresh CSV snapshot

Same output `run_pipeline(..., report_csv_path=...)` produces -- handy to
re-export after manually poking at the db, without re-running the pipeline.

In [ ]:
from protein_selector.core.report import write_report_csv

# write_report_csv(rows, Path("report_snapshot.csv"))